In [25]:
# Setup
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    default_headers={"HTTP-Referer": "http://localhost"}
)

model = "qwen/qwen-2.5-72b-instruct"

In [26]:
# Helper functions
import json

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, response):
    msg = response.choices[0].message
    messages.append({
        "role": "assistant",
        "content": msg.content,
        "tool_calls": msg.tool_calls if msg.tool_calls else None
    })

def add_tool_result(messages, tool_call_id, result):
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call_id,
        "content": str(result),
    })

def chat(messages, system=None, temperature=1.0, stop=None, tools=None):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages += messages

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": all_messages,
        "temperature": temperature,
        "stop": stop,
    }
    if tools:
        params["tools"] = tools

    return client.chat.completions.create(**params)

def text_from_message(response):
    return response.choices[0].message.content or ""

def get_tool_call(response):
    """Extract first tool call from response"""
    tool_calls = response.choices[0].message.tool_calls
    if not tool_calls:
        return None, None, None
    tool_call = tool_calls[0]
    return tool_call, tool_call.function.name, json.loads(tool_call.function.arguments)

In [27]:
# Tool functions
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(date.day,
            [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
             31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

def execute_tool(tool_name, tool_args):
    """Dispatcher - runs the correct function by name"""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_args)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_args)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_args)
    else:
        return f"Unknown tool: {tool_name}"

In [28]:
# tool schemas
get_current_datetime_schema = {
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string.",
        "parameters": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "Python strftime format string. Default is '%Y-%m-%d %H:%M:%S'.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    },
}

add_duration_to_datetime_schema = {
    "type": "function",
    "function": {
        "name": "add_duration_to_datetime",
        "description": "Adds a specified duration to a datetime string and returns the resulting datetime.",
        "parameters": {
            "type": "object",
            "properties": {
                "datetime_str": {"type": "string", "description": "The input datetime string."},
                "duration": {"type": "number", "description": "The amount of time to add. Defaults to 0."},
                "unit": {"type": "string", "description": "One of: seconds, minutes, hours, days, weeks, months, years."},
                "input_format": {"type": "string", "description": "Format string e.g. '%Y-%m-%d'."},
            },
            "required": ["datetime_str"],
        },
    },
}

set_reminder_schema = {
    "type": "function",
    "function": {
        "name": "set_reminder",
        "description": "Creates a timed reminder that will notify the user at the specified time.",
        "parameters": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "The reminder message text."},
                "timestamp": {"type": "string", "description": "ISO 8601 timestamp e.g. 2026-03-22T10:00:00."},
            },
            "required": ["content", "timestamp"],
        },
    },
}

batch_tool_schema = {
    "type": "function",
    "function": {
        "name": "batch_tool",
        "description": "Invoke multiple other tool calls simultaneously",
        "parameters": {
            "type": "object",
            "properties": {
                "invocations": {
                    "type": "array",
                    "description": "The tool calls to invoke",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string", "description": "The name of the tool to invoke"},
                            "arguments": {"type": "string", "description": "The arguments as a JSON string"},
                        },
                        "required": ["name", "arguments"],
                    },
                }
            },
            "required": ["invocations"],
        },
    },
}

ALL_TOOLS = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    batch_tool_schema,
]

print("✅ All schemas loaded")

✅ All schemas loaded


In [29]:
#  Basic tool call example (add_duration + 10 days)
today = datetime.now().strftime("%Y-%m-%d")

messages = []
add_user_message(messages, f"Today's date is {today}. What is today's date plus 10 days?")

response = chat(messages, tools=ALL_TOOLS)

tool_call, tool_name, tool_args = get_tool_call(response)
print(f"Tool called : {tool_name}")
print(f"Arguments  : {tool_args}")

result = execute_tool(tool_name, tool_args)
print(f"Result     : {result}")

Tool called : add_duration_to_datetime
Arguments  : {'datetime_str': '2026-03-22', 'duration': 10, 'unit': 'days', 'input_format': '%Y-%m-%d'}
Result     : Wednesday, April 01, 2026 12:00:00 AM


In [30]:
# Send tool result back and get final response
add_assistant_message(messages, response)
add_tool_result(messages, tool_call.id, result)

final_response = chat(messages, tools=ALL_TOOLS)
print(text_from_message(final_response))

The date 10 days after 2026-03-22 is Wednesday, April 01, 2026.


In [31]:
#  get_current_datetime example
messages = []
add_user_message(messages, "What is the exact current time formatted as HH:MM:SS?")

response = chat(messages, tools=ALL_TOOLS)

tool_call, tool_name, tool_args = get_tool_call(response)
print(f"Tool called : {tool_name}")
print(f"Arguments  : {tool_args}")

result = execute_tool(tool_name, tool_args)
print(f"Result     : {result}")

# Send result back and get final answer
add_assistant_message(messages, response)
add_tool_result(messages, tool_call.id, result)

final_response = chat(messages, tools=ALL_TOOLS)
print(text_from_message(final_response))

Tool called : get_current_datetime
Arguments  : {'date_format': '%H:%M:%S'}
Result     : 22:17:58
The exact current time is 22:17:58.
